In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.visualization
import named_arrays as na
import msfc_ccd

In [ ]:
sensor = msfc_ccd.TeledyneCCD230()

temperature = na.linspace(200, 300, axis="temperature", num=101) * u.K

dark_current = sensor.dark_current(temperature)

In [ ]:
axis_time = "time"

darks = msfc_ccd.fits.open(
    path=na.ScalarArray(
        ndarray=np.array([
            msfc_ccd.samples.path_dark_2s_esis1,
            msfc_ccd.samples.path_dark_12s_esis1,
        ]),
        axes=axis_time,
    ),
).taps

rate = msfc_ccd.dark.current(darks, axis_time).outputs

rate.to(u.DN / u.s).ndarray

In [ ]:
flats = msfc_ccd.fits.open(
    path=na.ScalarArray(
        ndarray=np.array([
            msfc_ccd.samples.path_led_esis1,
            msfc_ccd.samples.path_led_esis1_next,
        ]),
        axes=axis_time,
    ),
).taps

gain = msfc_ccd.gain.photon_transfer(flats, axis_time).outputs

rate_electrons = (rate * gain).to(u.electron / u.s)

rate_electrons.ndarray

In [ ]:
temperature_holder = (-50.3 * u.deg_C).to(u.K, equivalencies=u.temperature())

sensor.dark_current(temperature_holder)

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    na.plt.plot(
        temperature,
        dark_current,
        ax=ax,
        label="datasheet model",
    )
    ax.axvspan(
        temperature.ndarray.min(),
        230 * u.K,
        color="gray",
        alpha=0.2,
        label="outside the range of the model",
    )
    ax.scatter(
        np.full(rate_electrons.size, temperature_holder.value) * u.K,
        rate_electrons.ndarray.ravel(),
        color="tab:red",
        zorder=3,
        label="measured, each tap",
    )
    ax.set_yscale("log")
    ax.set_xlabel(f"temperature ({ax.get_xlabel()})")
    ax.set_ylabel(f"dark current ({ax.get_ylabel()})")
    ax2 = ax.secondary_xaxis(
        "top",
        functions=(lambda t: t - 273.15, lambda t: t + 273.15),
    )
    ax2.set_xlabel("temperature (deg C)")
    ax.legend();